In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed/model_train.csv")
val_df = pd.read_csv("../data/processed/model_val.csv")
test_df = pd.read_csv("../data/processed/model_test.csv")

c:\Users\loaye\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\loaye\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
train_df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V27,V28,Amount,Class,Hour,Day,Hour_sin,Hour_cos,Time_of_day,amount_log
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.133558,-0.021053,149.62,0,0,0,0.000000e+00,1.0,Night,5.014760
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.008983,0.014724,2.69,0,0,0,0.000000e+00,1.0,Night,1.305626
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.055353,-0.059752,378.66,0,0,0,0.000000e+00,1.0,Night,5.939276
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.062723,0.061458,123.50,0,0,0,0.000000e+00,1.0,Night,4.824306
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.219422,0.215153,69.99,0,0,0,0.000000e+00,1.0,Night,4.262539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198603,132904.0,-2.348563,-2.118430,0.968801,0.171939,1.476173,-0.066357,1.316844,-0.029965,0.564404,...,-0.234552,-0.040684,406.29,0,12,1,1.224647e-16,-1.0,Afternoon,6.009525
198604,132905.0,-0.473957,1.460497,-0.687928,-0.454381,0.375335,-1.071949,0.634231,0.306173,-0.096931,...,0.211713,0.084178,8.99,0,12,1,1.224647e-16,-1.0,Afternoon,2.301585
198605,132905.0,0.183706,0.752492,0.120348,-0.731923,0.533554,-1.237835,1.021313,-0.329895,-0.316811,...,-0.013347,0.024715,1.00,0,12,1,1.224647e-16,-1.0,Afternoon,0.693147
198606,132905.0,1.774140,-1.081432,-0.092484,0.194936,-0.816991,0.975553,-1.165119,0.434134,1.988660,...,0.030006,-0.027821,87.00,0,12,1,1.224647e-16,-1.0,Afternoon,4.477337


In [3]:
numeric_features = train_df.drop(columns=['Class']).select_dtypes(include=[np.number]).columns.tolist()
categorical_features = train_df.drop(columns=['Class']).select_dtypes(include=[object]).columns.tolist()

C:\Users\loaye\AppData\Local\Temp\ipykernel_4712\2758812355.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = train_df.drop(columns=['Class']).select_dtypes(include=[object]).columns.tolist()


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pipeline_logreg = Pipeline([
    ('preprocessor', preprocessor),
    ('logistic_regression', LogisticRegression(class_weight='balanced', max_iter=1000))
])

X_train = train_df.drop(columns=['Class'])
y_train = train_df['Class']
pipeline_logreg.fit(X_train, y_train)
X_test = test_df.drop(columns=['Class'])
y_test = test_df['Class']
logreg_pred = pipeline_logreg.predict(X_test)
print(classification_report(y_test, logreg_pred))
print(confusion_matrix(y_test, logreg_pred))
print(roc_auc_score(y_test, logreg_pred))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98     42507
           1       0.03      0.88      0.05        52

    accuracy                           0.96     42559
   macro avg       0.51      0.92      0.52     42559
weighted avg       1.00      0.96      0.98     42559

[[40896  1611]
 [    6    46]]
0.9233578722780501


In [6]:
logreg_probabilities = pipeline_logreg.predict_proba(X_test)[:,1]
logreg_probabilities

array([0.02134331, 0.0039195 , 0.10638063, ..., 0.02087004, 0.00959157,
       0.02261682])

In [7]:
from sklearn.ensemble import RandomForestClassifier

pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('random_forest', RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=8, random_state=42))
])
pipeline_rf.fit(X_train, y_train)
rf_pred = pipeline_rf.predict(X_test)
print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))
print(roc_auc_score(y_test, rf_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42507
           1       0.85      0.75      0.80        52

    accuracy                           1.00     42559
   macro avg       0.92      0.87      0.90     42559
weighted avg       1.00      1.00      1.00     42559

[[42500     7]
 [   13    39]]
0.8749176606206037


In [8]:
import xgboost as xgb

pipeline_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('xgboost', xgb.XGBClassifier(n_estimators=300, max_depth=8, scale_pos_weight=4, random_state=42))
])
pipeline_xgb.fit(X_train, y_train)
xgb_pred = pipeline_xgb.predict(X_test)
print(classification_report(y_test, xgb_pred))
print(confusion_matrix(y_test, xgb_pred))
print(roc_auc_score(y_test, xgb_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42507
           1       0.90      0.73      0.81        52

    accuracy                           1.00     42559
   macro avg       0.95      0.87      0.90     42559
weighted avg       1.00      1.00      1.00     42559

[[42503     4]
 [   14    38]]
0.8653375643106747


In [9]:
from sklearn.metrics import recall_score, precision_score, f1_score

comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Recall': [recall_score(y_test, logreg_pred), recall_score(y_test, rf_pred), recall_score(y_test, xgb_pred)],
    'Precision': [precision_score(y_test, logreg_pred), precision_score(y_test, rf_pred), precision_score(y_test, xgb_pred)],
    'F1 Score': [f1_score(y_test, logreg_pred), f1_score(y_test, rf_pred), f1_score(y_test, xgb_pred)],
    'ROC AUC': [roc_auc_score(y_test, logreg_pred), roc_auc_score(y_test, rf_pred), roc_auc_score(y_test, xgb_pred)]
})
comparison_df

,Model,Recall,Precision,F1 Score,ROC AUC
0,Logistic Regression,0.884615,0.027761,0.053833,0.923358
1,Random Forest,0.750000,0.847826,0.795918,0.874918
2,XGBoost,0.730769,0.904762,0.808511,0.865338


with no hyperparameter tuning, xgboost model gets the best precision but worst recall, while logistic regression gets best recall but a very bad precision score (flags most transactions as fraud when they're not). 

## Adding PR-AUC to the comparison

Since fraud is very rare in this dataset, PR-AUC (Precision-Recall AUC) is a better metric than ROC-AUC or accuracy. The PDF also says PR-AUC should be our main metric, so let's add it to the comparison table.

In [10]:
from sklearn.metrics import average_precision_score

logreg_proba = pipeline_logreg.predict_proba(X_test)[:, 1]
rf_proba = pipeline_rf.predict_proba(X_test)[:, 1]
xgb_proba = pipeline_xgb.predict_proba(X_test)[:, 1]

comparison_df['PR AUC'] = [
    average_precision_score(y_test, logreg_proba),
    average_precision_score(y_test, rf_proba),
    average_precision_score(y_test, xgb_proba),
]
comparison_df

,Model,Recall,Precision,F1 Score,ROC AUC,PR AUC
0,Logistic Regression,0.884615,0.027761,0.053833,0.923358,0.723276
1,Random Forest,0.750000,0.847826,0.795918,0.874918,0.778682
2,XGBoost,0.730769,0.904762,0.808511,0.865338,0.759471


## Class Imbalance Experiment

The dataset has 284,807 transactions but only 492 are fraud, so this is a big deal for this project. Here we compare 3 ways of handling it, using XGBoost as the test model:

1. **No handling** — train on the data as-is.
2. **Class weighting** — use `scale_pos_weight` so the model pays more attention to fraud.
3. **Random under-sampling** — remove some of the legit transactions so the classes are more balanced.


In [11]:
# Strategy 1: No handling at all
pipeline_xgb_none = Pipeline([
    ('preprocessor', preprocessor),
    ('xgboost', xgb.XGBClassifier(n_estimators=300, max_depth=8, random_state=42))
])
pipeline_xgb_none.fit(X_train, y_train)
pred_none = pipeline_xgb_none.predict(X_test)
proba_none = pipeline_xgb_none.predict_proba(X_test)[:, 1]

In [12]:
# Strategy 2: Class weighting (scale_pos_weight = ratio of legit to fraud in training data)
fraud_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print('scale_pos_weight used:', fraud_ratio)

pipeline_xgb_weighted = Pipeline([
    ('preprocessor', preprocessor),
    ('xgboost', xgb.XGBClassifier(n_estimators=300, max_depth=8, scale_pos_weight=fraud_ratio, random_state=42))
])
pipeline_xgb_weighted.fit(X_train, y_train)
pred_weighted = pipeline_xgb_weighted.predict(X_test)
proba_weighted = pipeline_xgb_weighted.predict_proba(X_test)[:, 1]

scale_pos_weight used: 541.6448087431694


In [13]:
# Strategy 3: Random under-sampling of the majority (legit) class
fraud_train = train_df[train_df['Class'] == 1]
legit_train = train_df[train_df['Class'] == 0].sample(n=len(fraud_train), random_state=42)

train_under = pd.concat([fraud_train, legit_train]).sample(frac=1, random_state=42)  # shuffle

X_train_under = train_under.drop(columns=['Class'])
y_train_under = train_under['Class']

print('Original training size:', len(train_df), '-> Under-sampled size:', len(train_under))

pipeline_xgb_under = Pipeline([
    ('preprocessor', preprocessor),
    ('xgboost', xgb.XGBClassifier(n_estimators=300, max_depth=8, random_state=42))
])
pipeline_xgb_under.fit(X_train_under, y_train_under)
pred_under = pipeline_xgb_under.predict(X_test)
proba_under = pipeline_xgb_under.predict_proba(X_test)[:, 1]

Original training size: 198608 -> Under-sampled size: 732


In [14]:
imbalance_comparison_df = pd.DataFrame({
    'Strategy': ['No handling', 'Class weighting', 'Random under-sampling'],
    'Recall': [recall_score(y_test, pred_none), recall_score(y_test, pred_weighted), recall_score(y_test, pred_under)],
    'Precision': [precision_score(y_test, pred_none), precision_score(y_test, pred_weighted), precision_score(y_test, pred_under)],
    'F1 Score': [f1_score(y_test, pred_none), f1_score(y_test, pred_weighted), f1_score(y_test, pred_under)],
    'PR AUC': [average_precision_score(y_test, proba_none), average_precision_score(y_test, proba_weighted), average_precision_score(y_test, proba_under)],
})
imbalance_comparison_df

,Strategy,Recall,Precision,F1 Score,PR AUC
0,No handling,0.519231,0.750000,0.613636,0.529480
1,Class weighting,0.750000,0.866667,0.804124,0.757378
2,Random under-sampling,0.903846,0.017716,0.034750,0.685819


**Findings:** Compare the table above. Usually, under-sampling boosts recall (catches more fraud) but hurts precision (more false alarms), while class weighting is a bit more balanced between the two. Write your own observation here once you run it on your data — this is exactly the kind of comparison the report needs.

## Hyperparameter Tuning (XGBoost)

XGBoost is our best-performing baseline, so this is the model we tune. We use `RandomizedSearchCV` (faster than a full grid search) and score on **average_precision** (PR-AUC), since that's our main metric.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'xgboost__n_estimators': [200, 300, 500],
    'xgboost__max_depth': [4, 6, 8, 10],
    'xgboost__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'xgboost__subsample': [0.7, 0.8, 1.0],
    'xgboost__colsample_bytree': [0.7, 0.8, 1.0],
}

pipeline_xgb_search = Pipeline([
    ('preprocessor', preprocessor),
    ('xgboost', xgb.XGBClassifier(scale_pos_weight=fraud_ratio, random_state=42))
])

random_search = RandomizedSearchCV(
    pipeline_xgb_search,
    param_distributions=param_grid,
    n_iter=20,
    scoring='average_precision',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

random_search.fit(X_train, y_train)
print('Best parameters:', random_search.best_params_)
print('Best CV PR-AUC:', random_search.best_score_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [ ]:
# Evaluate the tuned model on the untouched test set
best_xgb_pipeline = random_search.best_estimator_

tuned_pred = best_xgb_pipeline.predict(X_test)
tuned_proba = best_xgb_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, tuned_pred))
print('ROC AUC:', roc_auc_score(y_test, tuned_proba))
print('PR AUC:', average_precision_score(y_test, tuned_proba))

In [ ]:
# Save the full tuning search results (every combination that was tried)
xgb_tuning_results = pd.DataFrame(random_search.cv_results_)
xgb_tuning_results = xgb_tuning_results.sort_values('mean_test_score', ascending=False)
xgb_tuning_results.head(10)

## Saving Everything

Now we save the trained models (so the rest of the team can use them) and the results tables (so Member 4 can use them in the final comparison, and so we have evidence for the report).

In [ ]:
import os
import joblib

os.makedirs('../models/classical', exist_ok=True)
os.makedirs('../reports/model_results', exist_ok=True)

# Save the models
joblib.dump(pipeline_logreg, '../models/classical/logistic_regression.pkl')
joblib.dump(pipeline_rf, '../models/classical/random_forest.pkl')
joblib.dump(best_xgb_pipeline, '../models/classical/xgboost.pkl')  # save the TUNED version as our final XGBoost

# Save the results tables
comparison_df.to_csv('../reports/model_results/classical_models.csv', index=False)
imbalance_comparison_df.to_csv('../reports/model_results/imbalance_experiment.csv', index=False)
xgb_tuning_results.to_csv('../reports/model_results/xgboost_tuning.csv', index=False)

print('Models and results saved!')

## Summary

- Trained and compared **Logistic Regression**, **Random Forest**, and **XGBoost** (with PR-AUC as the main metric).
- Ran a class-imbalance experiment comparing no handling vs. class weighting vs. under-sampling.
- Tuned XGBoost's hyperparameters with `RandomizedSearchCV`, scored on PR-AUC.
- Saved the final models to `models/classical/` and the results tables to `reports/model_results/` so the rest of the team (especially Member 4) can use them.

**Next steps for the team:** hand these saved models + `classical_models.csv` over to Member 4, who will build the common evaluation framework (cost matrix, threshold tuning, error analysis) using them.